In [ ]:
# %% [markdown]
# # Exploratory Data Analysis - Fraud Detection System
# 
# This notebook performs exploratory data analysis on financial transaction data to identify patterns, anomalies, and features useful for fraud detection.

# %% [markdown]
# ## 1. Setup and Configuration

# %%
import warnings
warnings.filterwarnings('ignore')

# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from datetime import datetime, timedelta
import json
import yaml
from pathlib import Path
import sys
import os

# Add src to path for importing custom modules
sys.path.append('../src')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

print("Libraries imported successfully!")

# %% [markdown]
# ## 2. Load Configuration

# %%
# Load configuration
config_path = '../config/settings.yaml'

try:
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)
    print("Configuration loaded successfully!")
except Exception as e:
    print(f"Error loading config: {e}")
    config = {}

# %% [markdown]
# ## 3. Data Loading

# %%
class DataLoader:
    """Load and prepare data for EDA"""
    
    @staticmethod
    def load_sample_data():
        """Load sample transaction data"""
        # Generate synthetic data for demonstration
        np.random.seed(42)
        n_samples = 10000
        
        # Generate dates
        dates = pd.date_range(start='2024-01-01', end='2024-03-31', periods=n_samples)
        
        # Generate transaction data
        data = {
            'transaction_id': [f'TX{i:08d}' for i in range(n_samples)],
            'customer_id': [f'CUST{np.random.randint(10000, 99999)}' for _ in range(n_samples)],
            'amount': np.random.exponential(100, n_samples).clip(1, 10000),
            'transaction_type': np.random.choice(['purchase', 'withdrawal', 'transfer', 'online_payment'], 
                                                n_samples, p=[0.6, 0.2, 0.15, 0.05]),
            'merchant_category': np.random.choice(['retail', 'food', 'travel', 'entertainment', 'utilities', 'other'], 
                                                  n_samples, p=[0.3, 0.2, 0.15, 0.1, 0.15, 0.1]),
            'location': np.random.choice(['US', 'UK', 'DE', 'FR', 'CA', 'AU', 'JP', 'SG'], n_samples),
            'device_type': np.random.choice(['mobile', 'desktop', 'tablet'], n_samples, p=[0.6, 0.3, 0.1]),
            'hour_of_day': np.random.randint(0, 24, n_samples),
            'day_of_week': np.random.randint(0, 7, n_samples),
            'timestamp': dates,
            'is_fraud': np.random.choice([0, 1], n_samples, p=[0.97, 0.03])
        }
        
        df = pd.DataFrame(data)
        
        # Add some patterns to fraud
        fraud_mask = df['is_fraud'] == 1
        
        # Make fraud transactions larger on average
        df.loc[fraud_mask, 'amount'] = np.random.exponential(500, fraud_mask.sum()).clip(100, 20000)
        
        # Make fraud more likely at unusual hours
        df.loc[fraud_mask, 'hour_of_day'] = np.random.choice([2, 3, 4, 5], fraud_mask.sum())
        
        # Make fraud more likely from certain locations
        high_risk_locations = ['SG', 'JP', 'AU']
        df.loc[fraud_mask, 'location'] = np.random.choice(high_risk_locations + list(df['location'].unique()), 
                                                          fraud_mask.sum(), p=[0.4, 0.3, 0.3] + [0] * 5)
        
        # Make fraud more likely for certain transaction types
        high_risk_types = ['online_payment', 'transfer']
        df.loc[fraud_mask, 'transaction_type'] = np.random.choice(high_risk_types + list(df['transaction_type'].unique()), 
                                                                  fraud_mask.sum(), p=[0.6, 0.4] + [0] * 2)
        
        return df
    
    @staticmethod
    def load_real_data(filepath):
        """Load real transaction data if available"""
        try:
            if Path(filepath).exists():
                if filepath.endswith('.csv'):
                    df = pd.read_csv(filepath)
                elif filepath.endswith('.parquet'):
                    df = pd.read_parquet(filepath)
                else:
                    raise ValueError("Unsupported file format")
                print(f"Data loaded from {filepath}")
                return df
            else:
                print(f"File {filepath} not found, generating sample data")
                return DataLoader.load_sample_data()
        except Exception as e:
            print(f"Error loading data: {e}")
            return DataLoader.load_sample_data()

# %%
# Load data
data_loader = DataLoader()
try:
    df = data_loader.load_real_data('../data/raw/transactions.csv')
except:
    df = data_loader.load_sample_data()

print(f"Data shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nData types:")
print(df.dtypes)
print(f"\nFirst 5 rows:")
df.head()

# %% [markdown]
# ## 4. Data Overview

# %%
def overview_data(df):
    """Provide comprehensive data overview"""
    
    print("=" * 80)
    print("DATA OVERVIEW")
    print("=" * 80)
    
    # Basic info
    print(f"\n📊 Dataset Shape: {df.shape[0]} rows × {df.shape[1]} columns")
    
    # Missing values
    missing_values = df.isnull().sum()
    missing_percentage = (missing_values / len(df)) * 100
    
    print(f"\n🔍 Missing Values:")
    if missing_values.sum() == 0:
        print("  No missing values found!")
    else:
        missing_df = pd.DataFrame({
            'Missing Count': missing_values[missing_values > 0],
            'Percentage (%)': missing_percentage[missing_values > 0]
        })
        print(missing_df)
    
    # Duplicate rows
    duplicates = df.duplicated().sum()
    print(f"\n🔄 Duplicate Rows: {duplicates} ({duplicates/len(df)*100:.2f}%)")
    
    # Fraud distribution
    if 'is_fraud' in df.columns:
        fraud_counts = df['is_fraud'].value_counts()
        fraud_percentage = df['is_fraud'].value_counts(normalize=True) * 100
        
        print(f"\n🎯 Fraud Distribution:")
        print(f"  Genuine Transactions: {fraud_counts.get(0, 0):,} ({fraud_percentage.get(0, 0):.2f}%)")
        print(f"  Fraudulent Transactions: {fraud_counts.get(1, 0):,} ({fraud_percentage.get(1, 0):.2f}%)")
        print(f"  Class Imbalance Ratio: {fraud_counts.get(0, 1)/fraud_counts.get(1, 1):.2f}:1" if fraud_counts.get(1, 0) > 0 else "No fraud cases")
    
    # Data types
    print(f"\n📝 Data Types:")
    dtype_counts = df.dtypes.value_counts()
    for dtype, count in dtype_counts.items():
        print(f"  {dtype}: {count} columns")
    
    # Memory usage
    memory_mb = df.memory_usage(deep=True).sum() / (1024 ** 2)
    print(f"\n💾 Memory Usage: {memory_mb:.2f} MB")
    
    # Unique values per column
    print(f"\n🎲 Unique Values per Column:")
    unique_counts = df.nunique()
    for col in df.columns:
        if unique_counts[col] < 20:  # Show categorical columns
            print(f"  {col}: {unique_counts[col]} unique values - {df[col].unique()[:5]}")
        else:
            print(f"  {col}: {unique_counts[col]} unique values")
    
    return df

# %%
df = overview_data(df)

# %% [markdown]
# ## 5. Statistical Analysis

# %%
def statistical_analysis(df):
    """Perform statistical analysis on numerical columns"""
    
    print("=" * 80)
    print("STATISTICAL ANALYSIS")
    print("=" * 80)
    
    # Select numerical columns
    numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    
    if not numerical_cols:
        print("No numerical columns found!")
        return
    
    print(f"\n📈 Numerical Columns: {numerical_cols}")
    
    # Basic statistics
    print(f"\n📊 Basic Statistics:")
    stats_df = df[numerical_cols].describe().T
    stats_df['cv'] = stats_df['std'] / stats_df['mean']  # Coefficient of variation
    stats_df['iqr'] = stats_df['75%'] - stats_df['25%']  # Interquartile range
    stats_df['range'] = stats_df['max'] - stats_df['min']  # Range
    
    display(stats_df)
    
    # Skewness and kurtosis
    print(f"\n📉 Distribution Statistics:")
    dist_stats = pd.DataFrame({
        'Skewness': df[numerical_cols].skew(),
        'Kurtosis': df[numerical_cols].kurtosis()
    })
    display(dist_stats)
    
    # Correlation analysis
    print(f"\n🔗 Correlation Matrix:")
    corr_matrix = df[numerical_cols].corr()
    
    # Visualize correlation matrix
    plt.figure(figsize=(12, 8))
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
                center=0, square=True, linewidths=.5, cbar_kws={"shrink": .8})
    plt.title('Correlation Matrix of Numerical Features', fontsize=16, pad=20)
    plt.tight_layout()
    plt.show()
    
    # Fraud vs non-fraud statistics
    if 'is_fraud' in df.columns and 'amount' in df.columns:
        print(f"\n🎯 Amount Statistics by Fraud Status:")
        fraud_stats = df.groupby('is_fraud')['amount'].agg(['mean', 'std', 'min', 'median', 'max', 'count'])
        fraud_stats['percentage'] = fraud_stats['count'] / fraud_stats['count'].sum() * 100
        display(fraud_stats)
    
    return stats_df

# %%
stats_df = statistical_analysis(df)

# %% [markdown]
# ## 6. Visual Analysis

# %%
def create_visualizations(df):
    """Create comprehensive visualizations"""
    
    print("=" * 80)
    print("VISUAL ANALYSIS")
    print("=" * 80)
    
    # 1. Fraud Distribution
    print("\n1. 🎯 Fraud Distribution")
    
    if 'is_fraud' in df.columns:
        fig, axes = plt.subplots(1, 2, figsize=(15, 6))
        
        # Count plot
        sns.countplot(data=df, x='is_fraud', ax=axes[0])
        axes[0].set_title('Fraud vs Non-Fraud Transactions', fontsize=14)
        axes[0].set_xlabel('Is Fraud')
        axes[0].set_ylabel('Count')
        
        # Add percentage labels
        total = len(df)
        for p in axes[0].patches:
            percentage = f'{100 * p.get_height() / total:.1f}%'
            axes[0].annotate(percentage, 
                           (p.get_x() + p.get_width() / 2., p.get_height()),
                           ha='center', va='center', 
                           xytext=(0, 10), textcoords='offset points')
        
        # Pie chart
        fraud_counts = df['is_fraud'].value_counts()
        labels = ['Genuine', 'Fraud'] if 0 in fraud_counts.index and 1 in fraud_counts.index else fraud_counts.index
        axes[1].pie(fraud_counts.values, labels=labels, autopct='%1.1f%%', 
                   colors=['#2ecc71', '#e74c3c'], startangle=90)
        axes[1].set_title('Fraud Distribution (%)')
        
        plt.tight_layout()
        plt.show()
    
    # 2. Amount Distribution
    print("\n2. 💰 Amount Distribution")
    
    if 'amount' in df.columns:
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        
        # Histogram
        axes[0].hist(df['amount'], bins=50, edgecolor='black', alpha=0.7)
        axes[0].set_title('Amount Distribution', fontsize=14)
        axes[0].set_xlabel('Amount')
        axes[0].set_ylabel('Frequency')
        axes[0].axvline(df['amount'].mean(), color='red', linestyle='--', 
                       label=f"Mean: ${df['amount'].mean():.2f}")
        axes[0].legend()
        
        # Box plot
        axes[1].boxplot(df['amount'], vert=False)
        axes[1].set_title('Amount Box Plot', fontsize=14)
        axes[1].set_xlabel('Amount')
        
        # Log-transformed histogram
        if (df['amount'] > 0).all():
            axes[2].hist(np.log1p(df['amount']), bins=50, edgecolor='black', alpha=0.7)
            axes[2].set_title('Log-Transformed Amount Distribution', fontsize=14)
            axes[2].set_xlabel('log(Amount + 1)')
            axes[2].set_ylabel('Frequency')
        
        plt.tight_layout()
        plt.show()
    
    # 3. Fraud by Amount
    print("\n3. 🔍 Fraud Analysis by Amount")
    
    if 'is_fraud' in df.columns and 'amount' in df.columns:
        fig, axes = plt.subplots(1, 2, figsize=(15, 6))
        
        # Box plot by fraud status
        sns.boxplot(data=df, x='is_fraud', y='amount', ax=axes[0])
        axes[0].set_title('Amount Distribution by Fraud Status', fontsize=14)
        axes[0].set_xlabel('Is Fraud')
        axes[0].set_ylabel('Amount')
        
        # Violin plot
        sns.violinplot(data=df, x='is_fraud', y='amount', ax=axes[1])
        axes[1].set_title('Amount Density by Fraud Status', fontsize=14)
        axes[1].set_xlabel('Is Fraud')
        axes[1].set_ylabel('Amount')
        
        plt.tight_layout()
        plt.show()
    
    # 4. Time-based Analysis
    print("\n4. ⏰ Time-Based Analysis")
    
    time_cols = [col for col in df.columns if 'hour' in col.lower() or 'time' in col.lower() or 'date' in col.lower()]
    
    if time_cols:
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        axes = axes.ravel()
        
        for i, time_col in enumerate(time_cols[:4]):
            if i < 4:
                if 'hour' in time_col.lower():
                    # Hour of day analysis
                    hour_fraud_rate = df.groupby(time_col)['is_fraud'].mean() if 'is_fraud' in df.columns else None
                    
                    axes[i].bar(range(24), hour_fraud_rate if hour_fraud_rate is not None else 
                               df[time_col].value_counts().sort_index())
                    axes[i].set_title(f'{time_col.replace("_", " ").title()}', fontsize=12)
                    axes[i].set_xlabel(time_col.replace("_", " ").title())
                    axes[i].set_ylabel('Fraud Rate' if hour_fraud_rate is not None else 'Count')
                    axes[i].set_xticks(range(0, 24, 3))
        
        plt.tight_layout()
        plt.show()
    
    # 5. Categorical Features Analysis
    print("\n5. 📊 Categorical Features Analysis")
    
    categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
    
    if categorical_cols:
        fig, axes = plt.subplots(2, 3, figsize=(18, 10))
        axes = axes.ravel()
        
        for i, cat_col in enumerate(categorical_cols[:6]):
            if i < 6:
                # Get top 10 categories
                top_categories = df[cat_col].value_counts().nlargest(10).index
                df_filtered = df[df[cat_col].isin(top_categories)]
                
                if 'is_fraud' in df.columns:
                    # Fraud rate by category
                    fraud_by_cat = df_filtered.groupby(cat_col)['is_fraud'].mean().sort_values(ascending=False)
                    axes[i].bar(range(len(fraud_by_cat)), fraud_by_cat.values)
                    axes[i].set_xticks(range(len(fraud_by_cat)))
                    axes[i].set_xticklabels(fraud_by_cat.index, rotation=45, ha='right')
                    axes[i].set_ylabel('Fraud Rate')
                else:
                    # Count plot
                    sns.countplot(data=df_filtered, x=cat_col, ax=axes[i], order=top_categories)
                    axes[i].tick_params(axis='x', rotation=45)
                
                axes[i].set_title(f'{cat_col.replace("_", " ").title()}', fontsize=12)
                axes[i].set_xlabel('')
        
        # Hide empty subplots
        for i in range(len(categorical_cols[:6]), 6):
            axes[i].set_visible(False)
        
        plt.tight_layout()
        plt.show()
    
    # 6. Interactive Plotly Visualizations
    print("\n6. 📈 Interactive Visualizations")
    
    if 'is_fraud' in df.columns and 'amount' in df.columns:
        # Scatter plot: Amount vs Time (colored by fraud)
        if 'timestamp' in df.columns:
            fig = px.scatter(df.sample(1000) if len(df) > 1000 else df,
                           x='timestamp', y='amount',
                           color='is_fraud',
                           title='Amount vs Time (Colored by Fraud Status)',
                           labels={'is_fraud': 'Fraud'},
                           hover_data=['transaction_type', 'location'])
            fig.show()
        
        # 3D Scatter plot (if we have enough features)
        if all(col in df.columns for col in ['amount', 'hour_of_day', 'day_of_week']):
            fig = px.scatter_3d(df.sample(1000) if len(df) > 1000 else df,
                              x='amount', y='hour_of_day', z='day_of_week',
                              color='is_fraud',
                              title='3D View: Amount, Hour, Day (Colored by Fraud)',
                              labels={'is_fraud': 'Fraud'})
            fig.show()

# %%
create_visualizations(df)

# %% [markdown]
# ## 7. Anomaly Detection Analysis

# %%
def anomaly_detection_analysis(df):
    """Analyze anomalies and outliers in the data"""
    
    print("=" * 80)
    print("ANOMALY DETECTION ANALYSIS")
    print("=" * 80)
    
    if 'amount' not in df.columns:
        print("Amount column not found for anomaly detection")
        return
    
    # 1. Statistical Outlier Detection using IQR
    print("\n1. 📊 Statistical Outlier Detection (IQR Method)")
    
    Q1 = df['amount'].quantile(0.25)
    Q3 = df['amount'].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df[(df['amount'] < lower_bound) | (df['amount'] > upper_bound)]
    non_outliers = df[(df['amount'] >= lower_bound) & (df['amount'] <= upper_bound)]
    
    print(f"  Q1 (25th percentile): ${Q1:.2f}")
    print(f"  Q3 (75th percentile): ${Q3:.2f}")
    print(f"  IQR: ${IQR:.2f}")
    print(f"  Lower Bound: ${lower_bound:.2f}")
    print(f"  Upper Bound: ${upper_bound:.2f}")
    print(f"  Number of Outliers: {len(outliers):,} ({len(outliers)/len(df)*100:.2f}%)")
    print(f"  Number of Non-Outliers: {len(non_outliers):,}")
    
    # 2. Z-Score Analysis
    print("\n2. 📈 Z-Score Analysis")
    
    from scipy import stats
    
    df['amount_zscore'] = np.abs(stats.zscore(df['amount']))
    zscore_outliers = df[df['amount_zscore'] > 3]
    
    print(f"  Transactions with |Z-score| > 3: {len(zscore_outliers):,} ({len(zscore_outliers)/len(df)*100:.2f}%)")
    
    # 3. Compare outliers with fraud
    if 'is_fraud' in df.columns:
        print("\n3. 🎯 Outlier vs Fraud Analysis")
        
        outlier_fraud_overlap = len(outliers[outliers['is_fraud'] == 1])
        zscore_fraud_overlap = len(zscore_outliers[zscore_outliers['is_fraud'] == 1])
        
        print(f"  Fraud in IQR Outliers: {outlier_fraud_overlap:,} ({outlier_fraud_overlap/len(outliers)*100:.2f}% of outliers)")
        print(f"  Fraud in Z-score Outliers: {zscore_fraud_overlap:,} ({zscore_fraud_overlap/len(zscore_outliers)*100:.2f}% of z-score outliers)")
        
        # Visualization
        fig, axes = plt.subplots(1, 2, figsize=(15, 6))
        
        # IQR outliers vs fraud
        outlier_status = []
        fraud_in_outliers = len(outliers[outliers['is_fraud'] == 1])
        non_fraud_in_outliers = len(outliers[outliers['is_fraud'] == 0])
        fraud_in_normal = len(df[(~df.index.isin(outliers.index)) & (df['is_fraud'] == 1)])
        non_fraud_in_normal = len(df[(~df.index.isin(outliers.index)) & (df['is_fraud'] == 0)])
        
        axes[0].bar(['Outliers', 'Normal'], [len(outliers), len(df) - len(outliers)], 
                   color=['#e74c3c', '#3498db'])
        axes[0].set_title('Transaction Distribution: Outliers vs Normal', fontsize=14)
        axes[0].set_ylabel('Count')
        
        # Add text labels
        for i, v in enumerate([len(outliers), len(df) - len(outliers)]):
            axes[0].text(i, v, f'{v:,}', ha='center', va='bottom')
        
        # Fraud distribution in outliers
        axes[1].bar(['Fraud in Outliers', 'Normal in Outliers', 'Fraud in Normal', 'Normal in Normal'],
                   [fraud_in_outliers, non_fraud_in_outliers, fraud_in_normal, non_fraud_in_normal],
                   color=['#c0392b', '#e74c3c', '#2980b9', '#3498db'])
        axes[1].set_title('Fraud Distribution: Outliers vs Normal', fontsize=14)
        axes[1].set_ylabel('Count')
        axes[1].tick_params(axis='x', rotation=45)
        
        plt.tight_layout()
        plt.show()
    
    # 4. Clustering-based Anomaly Detection (simplified)
    print("\n4. 🔍 Clustering-based Anomaly Detection")
    
    from sklearn.cluster import DBSCAN
    from sklearn.preprocessing import StandardScaler
    
    # Prepare features for clustering
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if 'is_fraud' in numeric_cols:
        numeric_cols.remove('is_fraud')
    if 'amount_zscore' in numeric_cols:
        numeric_cols.remove('amount_zscore')
    
    if len(numeric_cols) >= 2:
        X = df[numeric_cols].fillna(0).values
        X_scaled = StandardScaler().fit_transform(X)
        
        # Apply DBSCAN
        dbscan = DBSCAN(eps=0.5, min_samples=10)
        clusters = dbscan.fit_predict(X_scaled)
        
        df['dbscan_cluster'] = clusters
        anomalies = df[df['dbscan_cluster'] == -1]  # -1 indicates anomalies in DBSCAN
        
        print(f"  DBSCAN Anomalies Detected: {len(anomalies):,} ({len(anomalies)/len(df)*100:.2f}%)")
        print(f"  Number of clusters (excluding anomalies): {len(set(clusters)) - (1 if -1 in clusters else 0)}")
        
        # Plot clusters if we have 2D data
        if len(numeric_cols) == 2:
            plt.figure(figsize=(10, 6))
            scatter = plt.scatter(X_scaled[:, 0], X_scaled[:, 1], c=clusters, cmap='viridis', alpha=0.6)
            plt.colorbar(scatter)
            plt.title('DBSCAN Clustering for Anomaly Detection', fontsize=14)
            plt.xlabel(numeric_cols[0])
            plt.ylabel(numeric_cols[1])
            plt.show()
    
    return df

# %%
df = anomaly_detection_analysis(df)

# %% [markdown]
# ## 8. Feature Engineering Insights

# %%
def feature_engineering_insights(df):
    """Generate insights for feature engineering"""
    
    print("=" * 80)
    print("FEATURE ENGINEERING INSIGHTS")
    print("=" * 80)
    
    # 1. Date/Time Features
    print("\n1. ⏰ Date/Time Feature Engineering")
    
    date_time_features = []
    
    if 'timestamp' in df.columns:
        # Extract datetime features
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        df['transaction_hour'] = df['timestamp'].dt.hour
        df['transaction_day'] = df['timestamp'].dt.day
        df['transaction_month'] = df['timestamp'].dt.month
        df['transaction_year'] = df['timestamp'].dt.year
        df['transaction_dayofweek'] = df['timestamp'].dt.dayofweek
        df['transaction_weekend'] = df['timestamp'].dt.dayofweek.isin([5, 6]).astype(int)
        
        date_time_features = ['transaction_hour', 'transaction_day', 'transaction_month', 
                             'transaction_year', 'transaction_dayofweek', 'transaction_weekend']
        
        print("  Extracted features:")
        for feature in date_time_features:
            print(f"    • {feature}")
    
    # 2. Transaction Amount Features
    print("\n2. 💰 Transaction Amount Features")
    
    amount_features = []
    
    if 'amount' in df.columns:
        # Create amount-based features
        df['amount_log'] = np.log1p(df['amount'])
        df['amount_sqrt'] = np.sqrt(df['amount'])
        df['amount_binned'] = pd.cut(df['amount'], bins=5, labels=['Very Low', 'Low', 'Medium', 'High', 'Very High'])
        
        # Statistical features (if we have customer data)
        if 'customer_id' in df.columns:
            df['customer_avg_amount'] = df.groupby('customer_id')['amount'].transform('mean')
            df['customer_amount_std'] = df.groupby('customer_id')['amount'].transform('std')
            df['amount_deviation'] = (df['amount'] - df['customer_avg_amount']) / df['customer_amount_std'].replace(0, 1)
        
        amount_features = ['amount_log', 'amount_sqrt', 'amount_binned']
        if 'customer_id' in df.columns:
            amount_features.extend(['customer_avg_amount', 'customer_amount_std', 'amount_deviation'])
        
        print("  Created features:")
        for feature in amount_features:
            print(f"    • {feature}")
    
    # 3. Behavioral Features
    print("\n3. 👤 Behavioral Features")
    
    behavioral_features = []
    
    if 'customer_id' in df.columns and 'timestamp' in df.columns:
        # Sort by customer and timestamp
        df = df.sort_values(['customer_id', 'timestamp'])
        
        # Time between transactions
        df['time_since_last_tx'] = df.groupby('customer_id')['timestamp'].diff().dt.total_seconds() / 3600  # in hours
        df['time_since_last_tx'] = df['time_since_last_tx'].fillna(24)  # Default 24 hours for first transaction
        
        # Transaction frequency
        df['tx_freq_1h'] = df.groupby('customer_id')['transaction_id'].rolling(window='1H', on='timestamp').count().values
        df['tx_freq_24h'] = df.groupby('customer_id')['transaction_id'].rolling(window='24H', on='timestamp').count().values
        
        behavioral_features = ['time_since_last_tx', 'tx_freq_1h', 'tx_freq_24h']
        
        print("  Created features:")
        for feature in behavioral_features:
            print(f"    • {feature}")
    
    # 4. Location-based Features
    print("\n4. 📍 Location-based Features")
    
    location_features = []
    
    if 'location' in df.columns and 'customer_id' in df.columns:
        # Location changes
        df['prev_location'] = df.groupby('customer_id')['location'].shift(1)
        df['location_changed'] = (df['location'] != df['prev_location']).astype(int)
        df['location_changed'] = df['location_changed'].fillna(0)
        
        # Location frequency
        df['location_frequency'] = df.groupby('location')['transaction_id'].transform('count')
        df['location_fraud_rate'] = df.groupby('location')['is_fraud'].transform('mean') if 'is_fraud' in df.columns else 0
        
        location_features = ['location_changed', 'location_frequency', 'location_fraud_rate']
        
        print("  Created features:")
        for feature in location_features:
            print(f"    • {feature}")
    
    # 5. Merchant/Transaction Type Features
    print("\n5. 🏪 Merchant-based Features")
    
    merchant_features = []
    
    if 'merchant_category' in df.columns:
        # Merchant risk scoring
        if 'is_fraud' in df.columns:
            merchant_fraud_rate = df.groupby('merchant_category')['is_fraud'].mean()
            df['merchant_risk_score'] = df['merchant_category'].map(merchant_fraud_rate)
        
        # Transaction type risk
        if 'transaction_type' in df.columns and 'is_fraud' in df.columns:
            type_fraud_rate = df.groupby('transaction_type')['is_fraud'].mean()
            df['transaction_type_risk'] = df['transaction_type'].map(type_fraud_rate)
        
        merchant_features = ['merchant_risk_score', 'transaction_type_risk']
        
        print("  Created features:")
        for feature in merchant_features:
            if feature in df.columns:
                print(f"    • {feature}")
    
    # 6. Device-based Features
    print("\n6. 📱 Device-based Features")
    
    device_features = []
    
    if 'device_type' in df.columns and 'customer_id' in df.columns:
        # Device changes
        df['prev_device'] = df.groupby('customer_id')['device_type'].shift(1)
        df['device_changed'] = (df['device_type'] != df['prev_device']).astype(int)
        df['device_changed'] = df['device_changed'].fillna(0)
        
        # Device frequency
        df['device_frequency'] = df.groupby('device_type')['transaction_id'].transform('count')
        
        device_features = ['device_changed', 'device_frequency']
        
        print("  Created features:")
        for feature in device_features:
            print(f"    • {feature}")
    
    # Summary
    all_new_features = date_time_features + amount_features + behavioral_features + \
                      location_features + merchant_features + device_features
    
    print(f"\n🎯 TOTAL NEW FEATURES CREATED: {len(all_new_features)}")
    print(f"\n📊 Dataset shape after feature engineering: {df.shape}")
    
    return df

# %%
df = feature_engineering_insights(df)

# %% [markdown]
# ## 9. Correlation with Fraud

# %%
def analyze_fraud_correlations(df):
    """Analyze correlations with fraud"""
    
    print("=" * 80)
    print("FRAUD CORRELATION ANALYSIS")
    print("=" * 80)
    
    if 'is_fraud' not in df.columns:
        print("No fraud label column found!")
        return
    
    # 1. Correlation matrix with fraud
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    
    if numeric_cols:
        # Calculate correlation with fraud
        fraud_correlations = df[numeric_cols].corr()['is_fraud'].sort_values(ascending=False)
        
        print("\n📈 Correlation with Fraud (Top 20):")
        display(fraud_correlations.head(20))
        
        # Visualize top correlations
        top_corr = fraud_correlations.drop('is_fraud').head(10)
        
        plt.figure(figsize=(12, 6))
        bars = plt.barh(range(len(top_corr)), top_corr.values)
        plt.yticks(range(len(top_corr)), top_corr.index)
        plt.xlabel('Correlation Coefficient')
        plt.title('Top 10 Features Correlated with Fraud', fontsize=14)
        
        # Color bars based on correlation strength
        for bar, corr in zip(bars, top_corr.values):
            if abs(corr) > 0.3:
                bar.set_color('red' if corr > 0 else 'green')
            elif abs(corr) > 0.1:
                bar.set_color('orange' if corr > 0 else 'lightgreen')
            else:
                bar.set_color('gray')
        
        plt.grid(axis='x', alpha=0.3)
        plt.tight_layout()
        plt.show()
    
    # 2. Mutual Information Score
    print("\n2. 🔍 Mutual Information with Fraud")
    
    from sklearn.feature_selection import mutual_info_classif
    
    # Select features for MI calculation
    features_for_mi = df.select_dtypes(include=[np.number]).drop(columns=['is_fraud'], errors='ignore')
    
    if not features_for_mi.empty:
        # Calculate mutual information
        mi_scores = mutual_info_classif(features_for_mi, df['is_fraud'], random_state=42)
        mi_series = pd.Series(mi_scores, index=features_for_mi.columns).sort_values(ascending=False)
        
        print("Top 10 Features by Mutual Information:")
        display(mi_series.head(10))
        
        # Visualize MI scores
        plt.figure(figsize=(12, 6))
        mi_series.head(15).plot(kind='barh')
        plt.xlabel('Mutual Information Score')
        plt.title('Top 15 Features by Mutual Information with Fraud', fontsize=14)
        plt.grid(axis='x', alpha=0.3)
        plt.tight_layout()
        plt.show()
    
    # 3. Feature Importance using Random Forest
    print("\n3. 🌲 Feature Importance (Random Forest)")
    
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import train_test_split
    
    # Prepare data
    X = df.select_dtypes(include=[np.number]).drop(columns=['is_fraud'], errors='ignore')
    y = df['is_fraud']
    
    if not X.empty and len(y.unique()) > 1:
        # Split data
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
        
        # Train Random Forest
        rf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
        rf.fit(X_train, y_train)
        
        # Get feature importance
        importance_df = pd.DataFrame({
            'feature': X.columns,
            'importance': rf.feature_importances_
        }).sort_values('importance', ascending=False)
        
        print("Top 20 Features by Random Forest Importance:")
        display(importance_df.head(20))
        
        # Visualize feature importance
        plt.figure(figsize=(12, 8))
        importance_df.head(15).plot(kind='barh', x='feature', y='importance', legend=False)
        plt.xlabel('Importance Score')
        plt.title('Top 15 Features by Random Forest Importance', fontsize=14)
        plt.grid(axis='x', alpha=0.3)
        plt.tight_layout()
        plt.show()
    
    return df

# %%
df = analyze_fraud_correlations(df)

# %% [markdown]
# ## 10. Key Findings and Recommendations

# %%
def generate_findings_and_recommendations(df):
    """Generate key findings and recommendations"""
    
    print("=" * 80)
    print("KEY FINDINGS AND RECOMMENDATIONS")
    print("=" * 80)
    
    findings = []
    recommendations = []
    
    # 1. Data Quality
    print("\n🔍 DATA QUALITY")
    print("-" * 40)
    
    missing_values = df.isnull().sum().sum()
    if missing_values == 0:
        findings.append("✅ Excellent data quality with no missing values")
    elif missing_values / df.size < 0.05:
        findings.append("⚠️  Minor missing values detected (<5%)")
        recommendations.append("Consider imputation for missing values")
    else:
        findings.append("❌ Significant missing values detected")
        recommendations.append("Implement robust data imputation strategy")
    
    duplicates = df.duplicated().sum()
    if duplicates == 0:
        findings.append("✅ No duplicate records found")
    else:
        findings.append(f"⚠️  {duplicates} duplicate records detected")
        recommendations.append("Remove duplicate transactions")
    
    # 2. Fraud Distribution
    print("\n🎯 FRAUD DISTRIBUTION")
    print("-" * 40)
    
    if 'is_fraud' in df.columns:
        fraud_rate = df['is_fraud'].mean()
        if fraud_rate < 0.01:
            findings.append(f"❌ Extremely imbalanced data: Only {fraud_rate*100:.2f}% fraud")
            recommendations.append("Use techniques like SMOTE, ADASYN, or class weights")
        elif fraud_rate < 0.05:
            findings.append(f"⚠️  Imbalanced data: {fraud_rate*100:.2f}% fraud")
            recommendations.append("Consider oversampling or cost-sensitive learning")
        else:
            findings.append(f"✅ Balanced fraud distribution: {fraud_rate*100:.2f}% fraud")
    
    # 3. Transaction Patterns
    print("\n💸 TRANSACTION PATTERNS")
    print("-" * 40)
    
    if 'amount' in df.columns:
        avg_amount = df['amount'].mean()
        max_amount = df['amount'].max()
        
        if max_amount > avg_amount * 100:
            findings.append(f"⚠️  Extreme values detected: Max amount (${max_amount:,.2f}) is {max_amount/avg_amount:.0f}x average")
            recommendations.append("Consider amount capping or robust scaling")
        else:
            findings.append(f"✅ Transaction amounts within reasonable range")
    
    # 4. Temporal Patterns
    print("\n⏰ TEMPORAL PATTERNS")
    print("-" * 40)
    
    time_cols = [col for col in df.columns if 'hour' in col.lower() or 'time' in col]
    if time_cols and 'is_fraud' in df.columns:
        for time_col in time_cols[:2]:  # Check first two time columns
            fraud_by_time = df.groupby(time_col)['is_fraud'].mean()
            if fraud_by_time.std() > fraud_by_time.mean() * 0.5:
                findings.append(f"⚠️  Strong temporal pattern in fraud detected for {time_col}")
                recommendations.append(f"Include {time_col} as important feature")
    
    # 5. Feature Insights
    print("\n📊 FEATURE INSIGHTS")
    print("-" * 40)
    
    # Check for highly correlated features
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    if len(numeric_cols) > 1:
        corr_matrix = df[numeric_cols].corr().abs()
        high_corr_pairs = []
        for i in range(len(corr_matrix.columns)):
            for j in range(i+1, len(corr_matrix.columns)):
                if corr_matrix.iloc[i, j] > 0.8:
                    high_corr_pairs.append((corr_matrix.columns[i], corr_matrix.columns[j], 
                                          corr_matrix.iloc[i, j]))
        
        if high_corr_pairs:
            findings.append(f"⚠️  {len(high_corr_pairs)} pairs of highly correlated features (>0.8)")
            recommendations.append("Consider feature selection to remove multicollinearity")
            for pair in high_corr_pairs[:3]:  # Show first 3 pairs
                print(f"    • {pair[0]} ↔ {pair[1]}: {pair[2]:.3f}")
        else:
            findings.append("✅ No severe multicollinearity detected")
    
    # 6. Anomaly Detection
    print("\n🔍 ANOMALY DETECTION")
    print("-" * 40)
    
    if 'amount' in df.columns:
        from scipy import stats
        z_scores = np.abs(stats.zscore(df['amount'].dropna()))
        outliers = (z_scores > 3).sum()
        
        if outliers > 0:
            findings.append(f"⚠️  {outliers} statistical outliers detected in transaction amounts")
            recommendations.append("Investigate outliers - could be potential fraud or data errors")
        else:
            findings.append("✅ No significant statistical outliers detected")
    
    # Print Findings
    print("\n📋 SUMMARY OF FINDINGS")
    print("-" * 40)
    for finding in findings:
        print(f"  {finding}")
    
    print("\n💡 RECOMMENDATIONS")
    print("-" * 40)
    if recommendations:
        for i, recommendation in enumerate(recommendations, 1):
            print(f"  {i}. {recommendation}")
    else:
        print("  ✅ No major issues detected. Data is ready for modeling!")
    
    # Additional modeling recommendations
    print("\n🚀 MODELING RECOMMENDATIONS")
    print("-" * 40)
    
    modeling_recs = [
        "1. Use ensemble methods (Random Forest, XGBoost) for better performance",
        "2. Implement proper cross-validation with stratification",
        "3. Focus on precision-recall metrics rather than accuracy",
        "4. Consider using Isolation Forest or Autoencoders for anomaly detection",
        "5. Implement real-time feature calculation for production",
        "6. Create a feedback loop for model retraining",
        "7. Monitor model drift and data quality continuously"
    ]
    
    for rec in modeling_recs:
        print(f"  {rec}")
    
    return findings, recommendations

# %%
findings, recommendations = generate_findings_and_recommendations(df)

# %% [markdown]
# ## 11. Export Results

# %%
def export_results(df, findings, recommendations):
    """Export EDA results and processed data"""
    
    print("=" * 80)
    print("EXPORTING RESULTS")
    print("=" * 80)
    
    # Create output directory
    output_dir = Path('../data/processed')
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # 1. Save processed data
    processed_path = output_dir / 'transactions_processed.csv'
    df.to_csv(processed_path, index=False)
    print(f"✅ Processed data saved to: {processed_path}")
    
    # 2. Save summary statistics
    summary_stats = df.describe().T
    summary_stats_path = output_dir / 'summary_statistics.csv'
    summary_stats.to_csv(summary_stats_path)
    print(f"✅ Summary statistics saved to: {summary_stats_path}")
    
    # 3. Save fraud analysis
    if 'is_fraud' in df.columns:
        fraud_analysis = pd.DataFrame({
            'total_transactions': [len(df)],
            'fraud_count': [df['is_fraud'].sum()],
            'fraud_rate': [df['is_fraud'].mean()],
            'avg_amount_fraud': [df[df['is_fraud'] == 1]['amount'].mean() if 'amount' in df.columns else None],
            'avg_amount_non_fraud': [df[df['is_fraud'] == 0]['amount'].mean() if 'amount' in df.columns else None]
        })
        fraud_path = output_dir / 'fraud_analysis.csv'
        fraud_analysis.to_csv(fraud_path, index=False)
        print(f"✅ Fraud analysis saved to: {fraud_path}")
    
    # 4. Save findings and recommendations
    findings_df = pd.DataFrame({
        'type': ['finding'] * len(findings) + ['recommendation'] * len(recommendations),
        'text': findings + recommendations
    })
    findings_path = output_dir / 'eda_findings.csv'
    findings_df.to_csv(findings_path, index=False)
    print(f"✅ Findings and recommendations saved to: {findings_path}")
    
    # 5. Save feature importance
    if 'is_fraud' in df.columns:
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        if 'is_fraud' in numeric_cols:
            numeric_cols.remove('is_fraud')
        
        if numeric_cols:
            # Calculate correlation with fraud
            fraud_corr = df[numeric_cols + ['is_fraud']].corr()['is_fraud'].drop('is_fraud')
            importance_df = pd.DataFrame({
                'feature': fraud_corr.index,
                'correlation_with_fraud': fraud_corr.values,
                'abs_correlation': np.abs(fraud_corr.values)
            }).sort_values('abs_correlation', ascending=False)
            
            importance_path = output_dir / 'feature_importance.csv'
            importance_df.to_csv(importance_path, index=False)
            print(f"✅ Feature importance saved to: {importance_path}")
    
    print(f"\n🎉 All results exported successfully to: {output_dir}")
    
    return {
        'processed_data_path': processed_path,
        'summary_stats_path': summary_stats_path,
        'findings_path': findings_path
    }

# %%
export_paths = export_results(df, findings, recommendations)

# %% [markdown]
# ## 12. Conclusion

# %%
print("=" * 80)
print("EDA COMPLETE - SUMMARY")
print("=" * 80)

print(f"\n📊 Dataset Analyzed: {df.shape[0]:,} transactions × {df.shape[1]} features")
print(f"🎯 Fraud Rate: {df['is_fraud'].mean()*100:.2f}%" if 'is_fraud' in df.columns else "No fraud labels")

print("\n🔑 Key Insights:")
print("  • Data is ready for modeling with minimal preprocessing needed")
print("  • Multiple features show strong correlation with fraud")
print("  • Feature engineering has created valuable new predictors")
print("  • Anomaly detection can complement supervised learning")

print("\n📈 Next Steps:")
print("  1. Proceed to model development notebook")
print("  2. Use engineered features for training")
print("  3. Implement proper validation strategies")
print("  4. Consider ensemble methods for better performance")

print("\n🎉 EDA process completed successfully!")